In [1]:
#import libraries
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score,classification_report,confusion_matrix

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [4]:
df=pd.read_csv("/content/drive/My Drive/Colab Notebooks/Machine Learning/Diabetes Risk & Lifestyle Factor.csv")
print("Dataset loaded successfully")

Dataset loaded successfully


In [5]:
df.head()

,Glucose,BMI,Insulin,Age,Gender,Diet_Type,Exercise_Frequency,Heredity,Smoking,Alcohol,Stress_Score,Sleep_Hours,Diabetes_Status
0,101.3,30.9,83.6,51.0,Male,Healthy,3-5 times/week,No,No,NaN,7.5,6.5,1.0
1,121.8,28.0,78.7,52.0,Female,Unhealthy,Daily,No,No,Moderate,6.9,6.6,1.0
2,121.2,27.2,79.3,51.0,NaN,Healthy,3-5 times/week,No,No,Low,5.4,6.8,1.0
3,121.9,22.8,93.1,46.0,Male,Moderate,Daily,No,No,NaN,4.0,6.3,0.0
4,156.3,NaN,77.3,42.0,Male,Unhealthy,1-2 times/week,No,No,NaN,6.5,5.6,1.0


In [7]:
df.shape

(5000, 13)

In [8]:
df.describe()

,Glucose,BMI,Insulin,Age,Stress_Score,Sleep_Hours,Diabetes_Status
count,4751.000000,4791.000000,4744.000000,4759.000000,4748.000000,4758.000000,4760.000000
mean,122.097558,26.621791,80.810835,44.285984,5.949284,6.481084,0.924580
std,32.301067,5.478449,18.897666,11.905504,1.954350,1.222833,0.264096
min,60.000000,14.000000,17.400000,18.000000,1.000000,3.000000,0.000000
25%,99.400000,22.900000,67.300000,36.000000,4.600000,5.600000,1.000000
50%,120.700000,26.500000,80.300000,44.000000,5.950000,6.500000,1.000000
75%,143.750000,30.300000,93.325000,52.000000,7.400000,7.300000,1.000000
max,238.000000,47.500000,153.700000,80.000000,10.000000,10.000000,1.000000


In [9]:
df.isnull().sum()

,0
Glucose,249
BMI,209
Insulin,256
Age,241
Gender,246
Diet_Type,230
Exercise_Frequency,230
Heredity,257
Smoking,252
Alcohol,2618


In [10]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 13 columns):
 #   Column              Non-Null Count  Dtype  
---  ------              --------------  -----  
 0   Glucose             4751 non-null   float64
 1   BMI                 4791 non-null   float64
 2   Insulin             4744 non-null   float64
 3   Age                 4759 non-null   float64
 4   Gender              4754 non-null   object 
 5   Diet_Type           4770 non-null   object 
 6   Exercise_Frequency  4770 non-null   object 
 7   Heredity            4743 non-null   object 
 8   Smoking             4748 non-null   object 
 9   Alcohol             2382 non-null   object 
 10  Stress_Score        4748 non-null   float64
 11  Sleep_Hours         4758 non-null   float64
 12  Diabetes_Status     4760 non-null   float64
dtypes: float64(7), object(6)
memory usage: 507.9+ KB


In [11]:
df.isnull().sum()

,0
Glucose,249
BMI,209
Insulin,256
Age,241
Gender,246
Diet_Type,230
Exercise_Frequency,230
Heredity,257
Smoking,252
Alcohol,2618


In [21]:
df.dropna(inplace=True)

In [22]:
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns
categorical_cols = df.select_dtypes(include=['object', 'category']).columns
print("Numerical Variables:")
print(list(numerical_cols))
print("\nCategorical Variables:")
print(list(categorical_cols))

Numerical Variables:
['Glucose', 'BMI', 'Insulin', 'Age', 'Stress_Score', 'Sleep_Hours', 'Diabetes_Status']

Categorical Variables:
['Gender', 'Diet_Type', 'Exercise_Frequency', 'Heredity', 'Smoking', 'Alcohol']


In [23]:
print("no.of duplicate rows: ",df.duplicated().sum())

no.of duplicate rows:  0


In [24]:
x=df.iloc[:,:-1]
y=df.iloc[:,-1]
print(x.head())
print(y.head())

    Glucose   BMI  Insulin   Age  Gender  Diet_Type Exercise_Frequency  \
1     121.8  28.0     78.7  52.0  Female  Unhealthy              Daily   
7     134.1  25.5     86.8  58.0    Male    Healthy              Daily   
8     113.4  20.8     72.5  52.0    Male  Unhealthy              Daily   
13    120.3  19.7     80.9  40.0    Male    Healthy     1-2 times/week   
14    148.3  28.5    107.2  58.0  Female    Healthy              Daily   

   Heredity Smoking   Alcohol  Stress_Score  Sleep_Hours  
1        No      No  Moderate           6.9          6.6  
7        No      No       Low           5.7          7.7  
8        No      No       Low           5.4          7.9  
13       No      No       Low           3.6          7.2  
14      Yes      No       Low           7.6          7.6  
1     1.0
7     1.0
8     1.0
13    1.0
14    1.0
Name: Diabetes_Status, dtype: float64


In [25]:
#split the data into training and testing
X_train,X_test,y_train,y_test=train_test_split(x,y,test_size=0.2,random_state=42)
# Identify categorical columns in X
categorical_cols_X = x.select_dtypes(include=['object', 'category']).columns
# Apply one-hot encoding to categorical columns in X_train and X_test
X_train = pd.get_dummies(X_train, columns=categorical_cols_X, drop_first=True)
X_test = pd.get_dummies(X_test, columns=categorical_cols_X, drop_first=True)
# Ensure that X_train and X_test have the same columns after one-hot encoding
X_train, X_test = X_train.align(X_test, join='outer', axis=1, fill_value=0)

In [26]:
from sklearn.preprocessing import StandardScaler
scaler=StandardScaler()
X_train=scaler.fit_transform(X_train)
X_test=scaler.transform(X_test)

In [27]:
from sklearn.svm import LinearSVC
#model buiding
model=LinearSVC(random_state=42)
model.fit(X_train,y_train)

LinearSVC(random_state=42)

In [28]:
y_pred=model.predict(X_test)
accuracy=accuracy_score(y_test,y_pred)
print("Accuracy:",accuracy)

Accuracy: 0.9391634980988594


In [29]:
confusion_mat=confusion_matrix(y_test,y_pred)
print("Confusion Matrix:")
print(confusion_mat)

Confusion Matrix:
[[  0  16]
 [  0 247]]


In [30]:
classification_rep=classification_report(y_test,y_pred)
print("Classification Report:")
print(classification_rep)

Classification Report:
              precision    recall  f1-score   support

         0.0       0.00      0.00      0.00        16
         1.0       0.94      1.00      0.97       247

    accuracy                           0.94       263
   macro avg       0.47      0.50      0.48       263
weighted avg       0.88      0.94      0.91       263



/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
